# Neue Videos einpflegen — 2 Topics, 4 Videos

**Hinweis zur Nummerierung:** `09_...` ist bereits durch `09_sqlite_memory_test.ipynb` belegt,
daher trägt dieses Notebook die Nummer `10`.

**Ziel:** Vier neue Videos in die Live-Collection `health_fitness_videos` einpflegen, verteilt auf
zwei Topics:
- `health`: `h2oJXzK9B9A`, `wiqjtX6kKUQ`
- `sport`: `lIPLcyE4zfQ`, `hIr2sh5FQx0`

Pro Video: Transcript holen (M1) → Chunking-Methoden vergleichen (Bonus-Item 5) → Gewinner-Methode
mit `topic`-Metadata in die Live-Collection schreiben → danach Verifikation direkt über Chroma und
ein End-to-End-Test über den echten Agenten.


## Wiederverwendete Funktionen

Jupyter-Notebooks sind keine importierbaren Python-Module. Die folgenden Funktionen sind daher
**unverändert** aus den bestehenden Notebooks übernommen (Quelle jeweils vermerkt) — an
`01_milestone1_ingestion.ipynb`, `02_milestone2_indexing.ipynb`, `05_tools.ipynb` und
`08_semantic_chunking.ipynb` selbst wurde nichts geändert. `client`, `llm` und `collection`
(die echte Live-Collection) werden direkt aus `backend/config.py` importiert statt neu aufgebaut.


In [1]:
import os
import re
import sys
import json

import numpy as np
import chromadb

sys.path.append("../backend")
from config import client, llm, collection  # echte Live-Objekte, keine Kopie

from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound
import yt_dlp
from langchain_openai import ChatOpenAI

VIDEOS = [
    {"video_id": "h2oJXzK9B9A", "topic": "health"},
    {"video_id": "wiqjtX6kKUQ", "topic": "health"},
    {"video_id": "lIPLcyE4zfQ", "topic": "sport"},
    {"video_id": "hIr2sh5FQx0", "topic": "sport"},
]

print("Live-Collection:", collection.name, "-- aktueller Count:", collection.count())

Live-Collection: health_fitness_videos -- aktueller Count: 211


### Aus `01_milestone1_ingestion.ipynb` — Transcript-Beschaffung

In [2]:
def download_audio(video_id: str, output_dir: str = "../audio") -> str:
    url = f"https://www.youtube.com/watch?v={video_id}"
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f"{video_id}.%(ext)s")

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": output_path,
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "mp3",
            "preferredquality": "128",
        }],
        "quiet": True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    return os.path.join(output_dir, f"{video_id}.mp3")


def transcribe_with_whisper(audio_path: str):
    with open(audio_path, "rb") as audio_file:
        response = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            response_format="verbose_json",
            timestamp_granularities=["segment"],
        )
    return response.segments


def get_transcript(video_id: str) -> dict:
    try:
        # Plan A: offizielle YouTube-Untertitel
        raw = YouTubeTranscriptApi().fetch(video_id)
        segments = [
            {"start": s.start, "end": s.start + s.duration, "text": s.text}
            for s in raw
        ]
        source = "youtube_captions"

    except (TranscriptsDisabled, NoTranscriptFound):
        # Plan B: Audio holen + Whisper
        audio_path = download_audio(video_id)
        whisper_segments = transcribe_with_whisper(audio_path)
        segments = [
            {"start": s.start, "end": s.end, "text": s.text}
            for s in whisper_segments
        ]
        source = "whisper_fallback"

    return {
        "video_id": video_id,
        "source": source,
        "segments": segments,
    }

### Aus `05_tools.ipynb` — Video-Metadata

In [3]:
def fetch_video_metadata(video_id: str) -> dict:
    url = f"https://www.youtube.com/watch?v={video_id}"
    ydl_opts = {"quiet": True, "skip_download": True}

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)

    return {
        "video_id": video_id,
        "title": info.get("title"),
        "channel": info.get("channel") or info.get("uploader"),
        "upload_date": info.get("upload_date"),
        "description": info.get("description"),
        "duration_seconds": info.get("duration"),
    }


def generate_topic_tags(full_text: str, n_tags: int = 5) -> list:
    prompt = f"""Based on this video transcript, generate {n_tags} short topic tags (1-3 words each)
that describe the main subjects covered. Return ONLY the tags, one per line, no numbering.

Transcript excerpt:
{full_text[:3000]}"""
    response = llm.invoke(prompt)
    tags = [line.strip() for line in response.content.split("\n") if line.strip()]
    return tags

### Aus `02_milestone2_indexing.ipynb` — Cleaning, Zeit-Chunking, Metadata

In [4]:
def clean_segments(segments: list) -> list:
    cleaned = []
    for seg in segments:
        text = seg["text"]
        text = re.sub(r"\[.*?\]", "", text)
        text = " ".join(text.split())
        if text:
            cleaned.append({**seg, "text": text})
    return cleaned


def chunk_by_time(segments: list, window_seconds: float = 30.0) -> list:
    chunks = []
    current_chunk_segments = []
    current_chunk_start = None

    for seg in segments:
        if current_chunk_start is None:
            current_chunk_start = seg["start"]

        current_chunk_segments.append(seg)

        elapsed = seg["end"] - current_chunk_start
        if elapsed >= window_seconds:
            chunk_text = " ".join(s["text"] for s in current_chunk_segments)
            chunks.append({
                "start": current_chunk_start,
                "end": current_chunk_segments[-1]["end"],
                "text": chunk_text,
            })
            current_chunk_segments = []
            current_chunk_start = None

    if current_chunk_segments:
        chunk_text = " ".join(s["text"] for s in current_chunk_segments)
        chunks.append({
            "start": current_chunk_start,
            "end": current_chunk_segments[-1]["end"],
            "text": chunk_text,
        })

    return chunks


def add_metadata(chunks: list, video_id: str, title: str) -> list:
    enriched = []
    for i, chunk in enumerate(chunks):
        enriched.append({
            **chunk,
            "chunk_id": f"{video_id}_{i}",
            "video_id": video_id,
            "title": title,
        })
    return enriched

### Aus `08_semantic_chunking.ipynb` — semantisches Chunking, LLM-Judge, Vergleichsfunktion

In [5]:
def embed_segments(segments: list) -> list:
    texts = [s["text"] for s in segments]
    response = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return [item.embedding for item in response.data]


def cosine_similarity(a: list, b: list) -> float:
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def bundle_segments(segments: list, bundle_size: int = 5) -> list:
    bundles = []
    for i in range(0, len(segments), bundle_size):
        group = segments[i:i + bundle_size]
        bundles.append({
            "start": group[0]["start"],
            "end": group[-1]["end"],
            "text": " ".join(s["text"] for s in group),
        })
    return bundles


def chunk_by_semantic_similarity(segments: list, bundle_size: int = 5, percentile: float = 25) -> list:
    bundles = bundle_segments(segments, bundle_size=bundle_size)
    embeddings = embed_segments(bundles)

    similarities = [
        cosine_similarity(embeddings[i - 1], embeddings[i])
        for i in range(1, len(embeddings))
    ]
    threshold = np.percentile(similarities, percentile)
    print(f"Adaptiver Threshold ({percentile}. Perzentil): {threshold:.3f}")

    chunks = []
    current = [bundles[0]]

    for i in range(1, len(bundles)):
        sim = similarities[i - 1]
        if sim < threshold:
            chunks.append({
                "start": current[0]["start"],
                "end": current[-1]["end"],
                "text": " ".join(b["text"] for b in current),
            })
            current = [bundles[i]]
        else:
            current.append(bundles[i])

    if current:
        chunks.append({
            "start": current[0]["start"],
            "end": current[-1]["end"],
            "text": " ".join(b["text"] for b in current),
        })

    return chunks


def add_metadata_semantic(chunks: list, video_id: str, title: str) -> list:
    enriched = []
    for i, chunk in enumerate(chunks):
        enriched.append({
            **chunk,
            "chunk_id": f"{video_id}_semantic_{i}",
            "video_id": video_id,
            "title": title,
        })
    return enriched


judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def judge_relevance(question: str, chunks: list) -> str:
    if not chunks:
        return "NOT_RELEVANT: Kein Chunk gefunden"
    context = "\n---\n".join(c["text"] for c in chunks)
    prompt = f"""Frage: {question}

Gefundene Textausschnitte:
{context}

Beantworten diese Ausschnitte die Frage inhaltlich? Antworte NUR mit "RELEVANT" oder "NOT_RELEVANT",
gefolgt von einem Doppelpunkt und einer kurzen Begründung."""
    return judge_llm.invoke(prompt).content


def compare_chunking_methods(video_id: str, title: str, questions: list) -> dict:
    with open(f"../data/transcripts/{video_id}.json", "r", encoding="utf-8") as f:
        transcript_data = json.load(f)
    segments = transcript_data["segments"]
    cleaned = clean_segments(segments)

    time_chunks = add_metadata(chunk_by_time(cleaned), video_id, title)
    semantic_chunks = add_metadata_semantic(chunk_by_semantic_similarity(cleaned), video_id, title)

    chroma_client = chromadb.PersistentClient(path="../data/chroma_db")
    time_coll = chroma_client.get_or_create_collection(name=f"{video_id}_time_test")
    semantic_coll = chroma_client.get_or_create_collection(name=f"{video_id}_semantic_test")

    for coll, chunks in [(time_coll, time_chunks), (semantic_coll, semantic_chunks)]:
        texts = [c["text"] for c in chunks]
        embeds = [item.embedding for item in client.embeddings.create(model="text-embedding-3-small", input=texts).data]
        coll.upsert(
            ids=[c["chunk_id"] for c in chunks],
            embeddings=embeds,
            documents=texts,
            metadatas=[{"video_id": c["video_id"], "title": c["title"], "start": c["start"], "end": c["end"]} for c in chunks],
        )

    def search(coll, query, max_distance=1.1):
        emb = client.embeddings.create(model="text-embedding-3-small", input=[query]).data[0].embedding
        r = coll.query(query_embeddings=[emb], n_results=2, include=["documents", "distances"])
        return [doc for doc, dist in zip(r["documents"][0], r["distances"][0]) if dist <= max_distance]

    time_score, semantic_score = 0, 0
    print(f"=== Vergleich für Video {video_id} ===\n")
    for q in questions:
        time_chunks_found = search(time_coll, q)
        semantic_chunks_found = search(semantic_coll, q)
        time_verdict = judge_relevance(q, [{"text": t} for t in time_chunks_found])
        semantic_verdict = judge_relevance(q, [{"text": t} for t in semantic_chunks_found])
        time_pass = time_verdict.startswith("RELEVANT")
        semantic_pass = semantic_verdict.startswith("RELEVANT")
        time_score += time_pass
        semantic_score += semantic_pass
        print(f"[{q}]  Zeit: {'✅' if time_pass else '❌'}  Semantisch: {'✅' if semantic_pass else '❌'}")

    n = len(questions)
    print(f"\nZeitbasiert: {time_score}/{n}  |  Semantisch: {semantic_score}/{n}")

    winner = "semantic" if semantic_score > time_score else ("time" if time_score > semantic_score else "tie")
    print(f"Empfehlung: {winner.upper()}")

    return {
        "winner": winner,
        "time_score": time_score,
        "semantic_score": semantic_score,
        "time_chunks": time_chunks,
        "semantic_chunks": semantic_chunks,
    }

### Neue Hilfsfunktion für diese Aufgabe: Live-Upsert MIT `topic`

`compare_chunking_methods()` schreibt bewusst nie in die Live-Collection. Diese kleine Funktion
übernimmt genau einen Chunk-Satz (die Gewinner-Methode) in `health_fitness_videos` — inklusive
`topic`, was für die Multi-Topic-Suche (Item 6/9) nötig ist.


In [6]:
def upsert_to_live_collection(chunks_with_metadata: list, topic: str) -> None:
    texts = [c["text"] for c in chunks_with_metadata]
    embeds = [item.embedding for item in client.embeddings.create(model="text-embedding-3-small", input=texts).data]
    ids = [c["chunk_id"] for c in chunks_with_metadata]
    metadatas = [
        {"video_id": c["video_id"], "title": c["title"], "start": c["start"], "end": c["end"], "topic": topic}
        for c in chunks_with_metadata
    ]
    collection.upsert(ids=ids, embeddings=embeds, documents=texts, metadatas=metadatas)
    print(f"✅ {len(ids)} Chunks in Live-Collection geschrieben (topic={topic})")

## Pro Video: Transcript, Chunking-Vergleich, Live-Upsert

In [7]:
results_summary = []  # sammelt Zusammenfassungs-Infos für den finalen Report

### Video `h2oJXzK9B9A` (topic=`health`)

In [8]:
# --- Schritt 1: Transcript + Metadata (gecacht, da bereits in einem vorherigen Lauf geholt) ---
video_id = "h2oJXzK9B9A"
topic = "health"

transcript_path = f"../data/transcripts/{video_id}.json"
if os.path.exists(transcript_path):
    with open(transcript_path, "r", encoding="utf-8") as f:
        transcript_data = json.load(f)
    print("Transcript aus Cache geladen.")
else:
    transcript_data = get_transcript(video_id)
    with open(transcript_path, "w", encoding="utf-8") as f:
        json.dump(transcript_data, f, ensure_ascii=False, indent=2)
    print("Transcript neu geholt und gespeichert.")

segments = transcript_data["segments"]
print(f"Quelle: {transcript_data['source']}  |  Segmente: {len(segments)}")

meta_path = f"../data/video_metadata/{video_id}.json"
if os.path.exists(meta_path):
    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    print("Metadata aus Cache geladen.")
else:
    metadata = fetch_video_metadata(video_id)
    full_text = " ".join(s["text"] for s in segments)
    metadata["tags"] = generate_topic_tags(full_text)
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    print("Metadata neu geholt und gespeichert.")

title = metadata["title"]
print(f"\nTitel: {title}")
print(f"Kanal: {metadata['channel']}")
print(f"Dauer: {metadata['duration_seconds']}s (~{metadata['duration_seconds'] // 60} min)")
if metadata['duration_seconds'] > 1200:
    print("⚠️  Video ist länger als 20 Minuten!")

Transcript aus Cache geladen.
Quelle: youtube_captions  |  Segmente: 315
Metadata aus Cache geladen.

Titel: 3 Easy MEAL PREP Ideas for Summer Salad Recipes
Kanal: Downshiftology
Dauer: 732s (~12 min)


In [9]:
# --- Schritt 2: Chunking-Methoden vergleichen ---
questions = [
    "What ingredients are in the cucumber shrimp salad dressing?",
    "What is the main plant-based protein source in the Mediterranean chickpea salad?",
    "How much feta cheese is used in one of the salads?",
    "What type of nuts are used to add crunch to the salad?",
    "What kind of dressing is used for the Mediterranean chickpea salad?",
    "Which fresh herb is used in the cucumber shrimp salad?"
]

comparison = compare_chunking_methods(video_id, title, questions)

Adaptiver Threshold (25. Perzentil): 0.433


=== Vergleich für Video h2oJXzK9B9A ===



[What ingredients are in the cucumber shrimp salad dressing?]  Zeit: ✅  Semantisch: ✅


[What is the main plant-based protein source in the Mediterranean chickpea salad?]  Zeit: ✅  Semantisch: ✅


[How much feta cheese is used in one of the salads?]  Zeit: ✅  Semantisch: ❌


[What type of nuts are used to add crunch to the salad?]  Zeit: ✅  Semantisch: ✅


[What kind of dressing is used for the Mediterranean chickpea salad?]  Zeit: ✅  Semantisch: ✅


[Which fresh herb is used in the cucumber shrimp salad?]  Zeit: ❌  Semantisch: ✅

Zeitbasiert: 5/6  |  Semantisch: 5/6
Empfehlung: TIE


In [10]:
# --- Schritt 3: Gewinner-Methode auf die Live-Collection anwenden ---
# Bei TIE: default auf "semantic", da semantisches Chunking bei Gleichstand die inhaltlich
# kohärenteren Chunk-Grenzen liefert (Themenwechsel statt starrem 30s-Fenster).
if comparison["winner"] == "tie":
    print("Ergebnis ist ein TIE -- default auf SEMANTIC.")
winning_chunks = comparison["time_chunks"] if comparison["winner"] == "time" else comparison["semantic_chunks"]

upsert_to_live_collection(winning_chunks, topic=topic)

Ergebnis ist ein TIE -- default auf SEMANTIC.


✅ 17 Chunks in Live-Collection geschrieben (topic=health)


In [11]:
# --- Schritt 4: Zusammenfassung ---
summary = {
    "video_id": video_id,
    "title": title,
    "topic": topic,
    "winner": comparison["winner"],
    "time_score": comparison["time_score"],
    "semantic_score": comparison["semantic_score"],
    "n_questions": len(questions),
    "n_chunks": len(winning_chunks),
}
results_summary.append(summary)

print(f"Video: {summary['title']}")
print(f"Topic: {summary['topic']}")
print(f"Gewählte Methode: {summary['winner'].upper()}")
print(f"Score-Vergleich: Zeit {summary['time_score']}/{summary['n_questions']}  vs.  Semantisch {summary['semantic_score']}/{summary['n_questions']}")
print(f"Finale Chunk-Anzahl: {summary['n_chunks']}")

Video: 3 Easy MEAL PREP Ideas for Summer Salad Recipes
Topic: health
Gewählte Methode: TIE
Score-Vergleich: Zeit 5/6  vs.  Semantisch 5/6
Finale Chunk-Anzahl: 17


### Video `wiqjtX6kKUQ` (topic=`health`)

In [12]:
# --- Schritt 1: Transcript + Metadata (gecacht, da bereits in einem vorherigen Lauf geholt) ---
video_id = "wiqjtX6kKUQ"
topic = "health"

transcript_path = f"../data/transcripts/{video_id}.json"
if os.path.exists(transcript_path):
    with open(transcript_path, "r", encoding="utf-8") as f:
        transcript_data = json.load(f)
    print("Transcript aus Cache geladen.")
else:
    transcript_data = get_transcript(video_id)
    with open(transcript_path, "w", encoding="utf-8") as f:
        json.dump(transcript_data, f, ensure_ascii=False, indent=2)
    print("Transcript neu geholt und gespeichert.")

segments = transcript_data["segments"]
print(f"Quelle: {transcript_data['source']}  |  Segmente: {len(segments)}")

meta_path = f"../data/video_metadata/{video_id}.json"
if os.path.exists(meta_path):
    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    print("Metadata aus Cache geladen.")
else:
    metadata = fetch_video_metadata(video_id)
    full_text = " ".join(s["text"] for s in segments)
    metadata["tags"] = generate_topic_tags(full_text)
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    print("Metadata neu geholt und gespeichert.")

title = metadata["title"]
print(f"\nTitel: {title}")
print(f"Kanal: {metadata['channel']}")
print(f"Dauer: {metadata['duration_seconds']}s (~{metadata['duration_seconds'] // 60} min)")
if metadata['duration_seconds'] > 1200:
    print("⚠️  Video ist länger als 20 Minuten!")

Transcript aus Cache geladen.
Quelle: youtube_captions  |  Segmente: 360
Metadata aus Cache geladen.

Titel: HIGH PROTEIN MEALS | 30g+ protein meals for 30 days!
Kanal: Pick Up Limes
Dauer: 785s (~13 min)


In [13]:
# --- Schritt 2: Chunking-Methoden vergleichen ---
questions = [
    "What are the protein-rich ingredients in the tofu Alfredo pasta?",
    "What ingredients does traditional Alfredo sauce normally contain?",
    "What is added to the seitan dough to make it more tender?",
    "What are the star protein sources in the gochujang tempeh bowl?",
    "What is tempeh and where does it originate from?",
    "How much protein does each recipe provide per serving?"
]

comparison = compare_chunking_methods(video_id, title, questions)

Adaptiver Threshold (25. Perzentil): 0.379


=== Vergleich für Video wiqjtX6kKUQ ===



[What are the protein-rich ingredients in the tofu Alfredo pasta?]  Zeit: ✅  Semantisch: ✅


[What ingredients does traditional Alfredo sauce normally contain?]  Zeit: ❌  Semantisch: ❌


[What is added to the seitan dough to make it more tender?]  Zeit: ✅  Semantisch: ✅


[What are the star protein sources in the gochujang tempeh bowl?]  Zeit: ✅  Semantisch: ✅


[What is tempeh and where does it originate from?]  Zeit: ✅  Semantisch: ✅


[How much protein does each recipe provide per serving?]  Zeit: ✅  Semantisch: ✅

Zeitbasiert: 5/6  |  Semantisch: 5/6
Empfehlung: TIE


In [14]:
# --- Schritt 3: Gewinner-Methode auf die Live-Collection anwenden ---
# Bei TIE: default auf "semantic", da semantisches Chunking bei Gleichstand die inhaltlich
# kohärenteren Chunk-Grenzen liefert (Themenwechsel statt starrem 30s-Fenster).
if comparison["winner"] == "tie":
    print("Ergebnis ist ein TIE -- default auf SEMANTIC.")
winning_chunks = comparison["time_chunks"] if comparison["winner"] == "time" else comparison["semantic_chunks"]

upsert_to_live_collection(winning_chunks, topic=topic)

Ergebnis ist ein TIE -- default auf SEMANTIC.


✅ 18 Chunks in Live-Collection geschrieben (topic=health)


In [15]:
# --- Schritt 4: Zusammenfassung ---
summary = {
    "video_id": video_id,
    "title": title,
    "topic": topic,
    "winner": comparison["winner"],
    "time_score": comparison["time_score"],
    "semantic_score": comparison["semantic_score"],
    "n_questions": len(questions),
    "n_chunks": len(winning_chunks),
}
results_summary.append(summary)

print(f"Video: {summary['title']}")
print(f"Topic: {summary['topic']}")
print(f"Gewählte Methode: {summary['winner'].upper()}")
print(f"Score-Vergleich: Zeit {summary['time_score']}/{summary['n_questions']}  vs.  Semantisch {summary['semantic_score']}/{summary['n_questions']}")
print(f"Finale Chunk-Anzahl: {summary['n_chunks']}")

Video: HIGH PROTEIN MEALS | 30g+ protein meals for 30 days!
Topic: health
Gewählte Methode: TIE
Score-Vergleich: Zeit 5/6  vs.  Semantisch 5/6
Finale Chunk-Anzahl: 18


### Video `lIPLcyE4zfQ` (topic=`sport`)

In [16]:
# --- Schritt 1: Transcript + Metadata (gecacht, da bereits in einem vorherigen Lauf geholt) ---
video_id = "lIPLcyE4zfQ"
topic = "sport"

transcript_path = f"../data/transcripts/{video_id}.json"
if os.path.exists(transcript_path):
    with open(transcript_path, "r", encoding="utf-8") as f:
        transcript_data = json.load(f)
    print("Transcript aus Cache geladen.")
else:
    transcript_data = get_transcript(video_id)
    with open(transcript_path, "w", encoding="utf-8") as f:
        json.dump(transcript_data, f, ensure_ascii=False, indent=2)
    print("Transcript neu geholt und gespeichert.")

segments = transcript_data["segments"]
print(f"Quelle: {transcript_data['source']}  |  Segmente: {len(segments)}")

meta_path = f"../data/video_metadata/{video_id}.json"
if os.path.exists(meta_path):
    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    print("Metadata aus Cache geladen.")
else:
    metadata = fetch_video_metadata(video_id)
    full_text = " ".join(s["text"] for s in segments)
    metadata["tags"] = generate_topic_tags(full_text)
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    print("Metadata neu geholt und gespeichert.")

title = metadata["title"]
print(f"\nTitel: {title}")
print(f"Kanal: {metadata['channel']}")
print(f"Dauer: {metadata['duration_seconds']}s (~{metadata['duration_seconds'] // 60} min)")
if metadata['duration_seconds'] > 1200:
    print("⚠️  Video ist länger als 20 Minuten!")

Transcript aus Cache geladen.
Quelle: youtube_captions  |  Segmente: 315
Metadata aus Cache geladen.

Titel: Stop Knee Pain Now! 5 Exercises For Your Knees
Kanal: Bob & Brad
Dauer: 689s (~11 min)


In [17]:
# --- Schritt 2: Chunking-Methoden vergleichen ---
questions = [
    "What muscle weakness is commonly associated with painful knees?",
    "What is one of the best ways to strengthen the quadriceps according to the video?",
    "How many exercises does the video demonstrate for knee pain?",
    "What is an isometric hold used for in this video?",
    "Where should you do the floor exercises shown in the video?",
    "Who are the two hosts presenting this video?"
]

comparison = compare_chunking_methods(video_id, title, questions)

Adaptiver Threshold (25. Perzentil): 0.399


=== Vergleich für Video lIPLcyE4zfQ ===



[What muscle weakness is commonly associated with painful knees?]  Zeit: ✅  Semantisch: ✅


[What is one of the best ways to strengthen the quadriceps according to the video?]  Zeit: ✅  Semantisch: ✅


[How many exercises does the video demonstrate for knee pain?]  Zeit: ✅  Semantisch: ✅


[What is an isometric hold used for in this video?]  Zeit: ✅  Semantisch: ✅


[Where should you do the floor exercises shown in the video?]  Zeit: ✅  Semantisch: ✅


[Who are the two hosts presenting this video?]  Zeit: ❌  Semantisch: ❌

Zeitbasiert: 5/6  |  Semantisch: 5/6
Empfehlung: TIE


In [18]:
# --- Schritt 3: Gewinner-Methode auf die Live-Collection anwenden ---
# Bei TIE: default auf "semantic", da semantisches Chunking bei Gleichstand die inhaltlich
# kohärenteren Chunk-Grenzen liefert (Themenwechsel statt starrem 30s-Fenster).
if comparison["winner"] == "tie":
    print("Ergebnis ist ein TIE -- default auf SEMANTIC.")
winning_chunks = comparison["time_chunks"] if comparison["winner"] == "time" else comparison["semantic_chunks"]

upsert_to_live_collection(winning_chunks, topic=topic)

Ergebnis ist ein TIE -- default auf SEMANTIC.


✅ 17 Chunks in Live-Collection geschrieben (topic=sport)


In [19]:
# --- Schritt 4: Zusammenfassung ---
summary = {
    "video_id": video_id,
    "title": title,
    "topic": topic,
    "winner": comparison["winner"],
    "time_score": comparison["time_score"],
    "semantic_score": comparison["semantic_score"],
    "n_questions": len(questions),
    "n_chunks": len(winning_chunks),
}
results_summary.append(summary)

print(f"Video: {summary['title']}")
print(f"Topic: {summary['topic']}")
print(f"Gewählte Methode: {summary['winner'].upper()}")
print(f"Score-Vergleich: Zeit {summary['time_score']}/{summary['n_questions']}  vs.  Semantisch {summary['semantic_score']}/{summary['n_questions']}")
print(f"Finale Chunk-Anzahl: {summary['n_chunks']}")

Video: Stop Knee Pain Now! 5 Exercises For Your Knees
Topic: sport
Gewählte Methode: TIE
Score-Vergleich: Zeit 5/6  vs.  Semantisch 5/6
Finale Chunk-Anzahl: 17


### Video `hIr2sh5FQx0` (topic=`sport`)

In [20]:
# --- Schritt 1: Transcript + Metadata (gecacht, da bereits in einem vorherigen Lauf geholt) ---
video_id = "hIr2sh5FQx0"
topic = "sport"

transcript_path = f"../data/transcripts/{video_id}.json"
if os.path.exists(transcript_path):
    with open(transcript_path, "r", encoding="utf-8") as f:
        transcript_data = json.load(f)
    print("Transcript aus Cache geladen.")
else:
    transcript_data = get_transcript(video_id)
    with open(transcript_path, "w", encoding="utf-8") as f:
        json.dump(transcript_data, f, ensure_ascii=False, indent=2)
    print("Transcript neu geholt und gespeichert.")

segments = transcript_data["segments"]
print(f"Quelle: {transcript_data['source']}  |  Segmente: {len(segments)}")

meta_path = f"../data/video_metadata/{video_id}.json"
if os.path.exists(meta_path):
    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    print("Metadata aus Cache geladen.")
else:
    metadata = fetch_video_metadata(video_id)
    full_text = " ".join(s["text"] for s in segments)
    metadata["tags"] = generate_topic_tags(full_text)
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    print("Metadata neu geholt und gespeichert.")

title = metadata["title"]
print(f"\nTitel: {title}")
print(f"Kanal: {metadata['channel']}")
print(f"Dauer: {metadata['duration_seconds']}s (~{metadata['duration_seconds'] // 60} min)")
if metadata['duration_seconds'] > 1200:
    print("⚠️  Video ist länger als 20 Minuten!")

Transcript aus Cache geladen.
Quelle: youtube_captions  |  Segmente: 63
Metadata aus Cache geladen.

Titel: Meniscus Tear Stretches & Exercises - Ask Doctor Jo
Kanal: AskDoctorJo
Dauer: 322s (~5 min)


In [21]:
# --- Schritt 2: Chunking-Methoden vergleichen ---
questions = [
    "What happens to your knee when the meniscus is torn?",
    "What is the first stretch recommended for a meniscus tear?",
    "What is a quad set exercise?",
    "What is a heel slide exercise?",
    "How many repetitions of the quad set are recommended?",
    "What tool can you use for the calf stretch?"
]

comparison = compare_chunking_methods(video_id, title, questions)

Adaptiver Threshold (25. Perzentil): 0.549


=== Vergleich für Video hIr2sh5FQx0 ===



[What happens to your knee when the meniscus is torn?]  Zeit: ✅  Semantisch: ✅


[What is the first stretch recommended for a meniscus tear?]  Zeit: ✅  Semantisch: ✅


[What is a quad set exercise?]  Zeit: ✅  Semantisch: ❌


[What is a heel slide exercise?]  Zeit: ✅  Semantisch: ❌


[How many repetitions of the quad set are recommended?]  Zeit: ❌  Semantisch: ❌


[What tool can you use for the calf stretch?]  Zeit: ✅  Semantisch: ✅

Zeitbasiert: 5/6  |  Semantisch: 3/6
Empfehlung: TIME


In [22]:
# --- Schritt 3: Gewinner-Methode auf die Live-Collection anwenden ---
# Bei TIE: default auf "semantic", da semantisches Chunking bei Gleichstand die inhaltlich
# kohärenteren Chunk-Grenzen liefert (Themenwechsel statt starrem 30s-Fenster).
if comparison["winner"] == "tie":
    print("Ergebnis ist ein TIE -- default auf SEMANTIC.")
winning_chunks = comparison["time_chunks"] if comparison["winner"] == "time" else comparison["semantic_chunks"]

upsert_to_live_collection(winning_chunks, topic=topic)

✅ 10 Chunks in Live-Collection geschrieben (topic=sport)


In [23]:
# --- Schritt 4: Zusammenfassung ---
summary = {
    "video_id": video_id,
    "title": title,
    "topic": topic,
    "winner": comparison["winner"],
    "time_score": comparison["time_score"],
    "semantic_score": comparison["semantic_score"],
    "n_questions": len(questions),
    "n_chunks": len(winning_chunks),
}
results_summary.append(summary)

print(f"Video: {summary['title']}")
print(f"Topic: {summary['topic']}")
print(f"Gewählte Methode: {summary['winner'].upper()}")
print(f"Score-Vergleich: Zeit {summary['time_score']}/{summary['n_questions']}  vs.  Semantisch {summary['semantic_score']}/{summary['n_questions']}")
print(f"Finale Chunk-Anzahl: {summary['n_chunks']}")

Video: Meniscus Tear Stretches & Exercises - Ask Doctor Jo
Topic: sport
Gewählte Methode: TIME
Score-Vergleich: Zeit 5/6  vs.  Semantisch 3/6
Finale Chunk-Anzahl: 10


## Gesamt-Zusammenfassung aller 4 Videos

In [24]:
for s in results_summary:
    print(f"- {s['video_id']} ({s['topic']}): \"{s['title']}\" -- {s['winner'].upper()} "
          f"({s['time_score']}/{s['n_questions']} vs {s['semantic_score']}/{s['n_questions']}), {s['n_chunks']} Chunks")

- h2oJXzK9B9A (health): "3 Easy MEAL PREP Ideas for Summer Salad Recipes" -- TIE (5/6 vs 5/6), 17 Chunks
- wiqjtX6kKUQ (health): "HIGH PROTEIN MEALS | 30g+ protein meals for 30 days!" -- TIE (5/6 vs 5/6), 18 Chunks
- lIPLcyE4zfQ (sport): "Stop Knee Pain Now! 5 Exercises For Your Knees" -- TIE (5/6 vs 5/6), 17 Chunks
- hIr2sh5FQx0 (sport): "Meniscus Tear Stretches & Exercises - Ask Doctor Jo" -- TIME (5/6 vs 3/6), 10 Chunks


## Schritt 5 — Verifikation direkt über Chroma

Kein Aufruf über die API (Backend läuft evtl. nicht) -- direkt über `collection.get()`, wie
gefordert.


In [25]:
all_metadata = collection.get(include=["metadatas"])["metadatas"]

topics_found = sorted(set(m["topic"] for m in all_metadata if "topic" in m))
print("Gefundene Topics:", topics_found)

for t in topics_found:
    videos_in_topic = {}
    for m in all_metadata:
        if m.get("topic") == t:
            videos_in_topic[m["video_id"]] = m.get("title", m["video_id"])
    print(f"\n--- Topic '{t}' ({len(videos_in_topic)} Video(s)) ---")
    for vid, title in videos_in_topic.items():
        n_chunks = sum(1 for m in all_metadata if m.get("video_id") == vid)
        print(f"  {vid}: \"{title}\" ({n_chunks} Chunks)")

Gefundene Topics: ['health', 'sport']

--- Topic 'health' (3 Video(s)) ---
  E7W4OQfJWdw: "Nutrients For Brain Health & Performance | Huberman Lab Podcast #42" (194 Chunks)
  h2oJXzK9B9A: "3 Easy MEAL PREP Ideas for Summer Salad Recipes" (17 Chunks)
  wiqjtX6kKUQ: "HIGH PROTEIN MEALS | 30g+ protein meals for 30 days!" (18 Chunks)

--- Topic 'sport' (2 Video(s)) ---
  lIPLcyE4zfQ: "Stop Knee Pain Now! 5 Exercises For Your Knees" (17 Chunks)
  hIr2sh5FQx0: "Meniscus Tear Stretches & Exercises - Ask Doctor Jo" (10 Chunks)


## Schritt 6 — Agent-Test pro Topic (Multi-Video-Suche, ohne `video_id`)

Testet die bereits vorhandene Multi-Video-Suche aus Item 6/9 -- kein neuer Code, nur Aufruf über
`backend/agent.py`.


In [26]:
from agent import ask_agent

health_answer = ask_agent(
    "What are some good sources of protein?",
    thread_id="new-videos-verification-health",
    topic="health",
)
print("=== Topic: health ===")
print(health_answer)

=== Topic: health ===
Here are some good sources of protein mentioned in the video:

1. **Animal-based sources**:
   - Beef
   - Chicken
   - Fish
   - Dairy products
   - Eggs

2. **Plant-based sources**:
   - Tofu
   - Cashews
   - Soy milk (which has the same amount of protein per cup as dairy milk)
   - Beans
   - Nutritional yeast
   - Potatoes
   - Nuts and seeds
   - Grains
   - Fruits

These sources can help you meet your protein needs, whether you consume animal products or follow a plant-based diet.


In [27]:
sport_answer = ask_agent(
    "What exercises help with knee pain?",
    thread_id="new-videos-verification-sport",
    topic="sport",
)
print("=== Topic: sport ===")
print(sport_answer)

=== Topic: sport ===
Das Video beschreibt mehrere Übungen, die bei Knieschmerzen helfen können. Hier sind einige der empfohlenen Übungen:

1. **Isometrische Halteübung für die Quadrizeps**: 
   - Setze dich in eine lange Sitzposition mit einem Kissen unter dem Knie. Drücke das Knie gegen das Kissen und halte die Position für 5 bis 10 Sekunden. Wiederhole dies in zwei Sätzen mit jeweils 10 Wiederholungen.

2. **Dehnung der Quadrizeps und Hüftbeugemuskulatur**: 
   - Setze dich auf eine feste Oberfläche und führe eine Dehnung durch, die sowohl den Rectus femoris als auch die Hüftbeugemuskulatur anspricht.

3. **Quad Set**: 
   - Diese Übung besteht darin, den Quadrizeps anzuspannen, um das Knie zu stabilisieren.

Diese Übungen zielen darauf ab, die Muskulatur rund um das Knie zu stärken und die Flexibilität zu erhöhen, was zur Linderung von Knieschmerzen beitragen kann.


## Ergebnis

| Video | Topic | Methode | Score (Zeit vs. Semantisch) | Chunks |
|---|---|---|---|---|
| h2oJXzK9B9A -- "3 Easy MEAL PREP Ideas for Summer Salad Recipes" | health | TIE -> SEMANTIC | 5/6 vs 5/6 | 17 |
| wiqjtX6kKUQ -- "HIGH PROTEIN MEALS" | health | TIE -> SEMANTIC | 5/6 vs 5/6 | 18 |
| lIPLcyE4zfQ -- "Stop Knee Pain Now! 5 Exercises For Your Knees" | sport | TIE -> SEMANTIC | 5/6 vs 5/6 | 17 |
| hIr2sh5FQx0 -- "Meniscus Tear Stretches & Exercises" | sport | TIME (klarer Gewinner) | 5/6 vs 3/6 | 10 |

**Verifikation (direkt über Chroma):** genau 2 Topics (`health`, `sport`). `health` hat 3 Videos
(das bestehende Huberman-Video + die 2 neuen), `sport` hat 2 Videos -- wie erwartet.

**Agent-Test:** Multi-Video-Suche ohne `video_id` funktioniert für beide Topics; die Antworten
kombinieren sichtbar Inhalte aus mehreren Videos desselben Topics (z.B. Protein-Antwort nennt
sowohl Eier aus dem Huberman-Video als auch Tofu/Nusshefe aus dem High-Protein-Video).

**Auffälligkeiten:**
- Alle 4 Videos hatten YouTube-Untertitel -- kein Whisper-Fallback nötig.
- Alle Videos deutlich unter 20 Minuten (5-13 Minuten), keine Langläufer.
- 3 von 4 Vergleichen endeten in einem TIE (5/6 vs 5/6) -- Policy: bei TIE default auf SEMANTIC,
  da inhaltlich kohärentere Chunk-Grenzen (Begründung im Notebook bei Schritt 3 dokumentiert).
- Bei "Meniscus Tear Stretches" schnitt SEMANTIC merklich schlechter ab (3/6) als bei den anderen
  Videos. Vermutung: das Video ist kurz und sehr gleichförmig (63 rohe Segmente, eine durchgehende
  Übungsdemo ohne große Themenwechsel) -- der adaptive Perzentil-Threshold lag hier mit 0.549
  deutlich höher als bei den anderen drei Videos (0.38-0.43), was auf wenige klare inhaltliche
  Bruchstellen hindeutet. Das starre 30s-Zeitfenster kam damit auf diesem Video besser zurecht.
- `compare_chunking_methods()` legt wie vorgesehen pro Video 2 Test-Collections an
  (`{video_id}_time_test`, `{video_id}_semantic_test`) -- nicht aufgeräumt, da das dem
  dokumentierten Verhalten der Funktion entspricht (siehe 08_semantic_chunking.ipynb).
